In [1]:
import argparse
import json
from pathlib import Path
import os
import time
from datetime import timedelta
import sys
sys.path.append("../..")
import glob
import numpy as np
import csv
import torch
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F
from monai import transforms
from monai.data import CacheDataset, DataLoader, ThreadDataLoader
from monai.data.utils import pad_list_data_collate
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import random
from monai.utils import first, set_determinism

from monai.inferers import LatentDiffusionInferer
from monai.networks.nets import DiffusionModelUNet, AutoencoderKL
from monai.networks.schedulers import DDPMScheduler

from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist

import utils.custom_transforms as custom_transforms
from utils.utils import *
import AnoDDPM.simplex as simplex
import utils.simplex_ddpm as simplex_ddpm

import nibabel as nib

import matplotlib.pyplot as plt

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    RandAffined,
    RandScaleCropd,
    ResizeWithPadOrCropd,
    ScaleIntensityRangeD,
    RandFlipd,
    Lambdad,
)

In [ ]:
def setup_ddp(rank, world_size):
    print(f"Running DDP LDM training on rank {rank}/world_size {world_size}.")
    print(f"Initing to IP {os.environ['MASTER_ADDR']}")
    dist.init_process_group(
        backend="nccl", init_method="env://", timeout=timedelta(seconds=36000), rank=rank, world_size=world_size
    )  # gloo, nccl
    dist.barrier()
    device = torch.device(f"cuda:{rank}")
    return dist, device


In [ ]:
ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"

In [4]:
config_dict = json.load(open(ROOT_DIR+"AnoDiffExperiments/experiment_2/exp_2_7/config.json", "r"))
args = argparse.Namespace(**config_dict)


In [5]:

# ----------------- SETUP ----------------- #

EXPERIMENT_NAME = args.experiment_name
SUB_EXPERIMENT_NAME = args.sub_experiment_name
MODELS_DIR = ROOT_DIR+f"AnoDiffExperiments/{EXPERIMENT_NAME}/{SUB_EXPERIMENT_NAME}/models/"
os.makedirs(MODELS_DIR, exist_ok=True)

ddp_bool = False  # whether to use distributed data parallel

if ddp_bool:
    rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    dist, device = setup_ddp(rank, world_size)
else:
    rank = 0
    world_size = 1
    device = 0

torch.cuda.set_device(device)
print(f"Using {device}")

torch.backends.cudnn.benchmark = True
torch.set_num_threads(torch.get_num_threads()) 
torch.autograd.set_detect_anomaly(False)


Using 0


### Data

Healthy test reconstruction

In [6]:

# ----------------- DATASET AND DATALOADER ----------------- #
test_reconstruction_csv = os.path.join(ROOT_DIR, f"AnoDiffExperiments/data_splits_lists/{args.dataset["name"]}/test.csv")
test_reconstruction_images_path = []

with open(test_reconstruction_csv, mode='r') as file:
    reader = csv.reader(file)
    for line in tqdm(reader):
        #print(line)
        test_reconstruction_images_path.append(ROOT_DIR+line[0])


test_reconstruction_datalist = test_reconstruction_images_path

batch_size = args.autoencoder_train["batch_size"]
num_workers = args.autoencoder_train["num_workers"]


243it [00:00, 117488.86it/s]


In [7]:

# Validation transforms
test_reconstruction_transforms = define_instance(args, "val_transforms")

# Update datalists to use image paths directly (not dictionaries)
test_reconstruction_datalist = test_reconstruction_images_path

# Create datasets
test_reconstruction_ds = CacheDataset(data=test_reconstruction_datalist[:batch_size], transform=test_reconstruction_transforms) #TODO

# Create samplers and dataloaders (as in your original code)
if ddp_bool:
    test_reconstruction_sampler = torch.utils.data.distributed.DistributedSampler(test_reconstruction_ds, num_replicas=world_size, rank=rank)
else:
    test_reconstruction_sampler = None
    
test_reconstruction_loader = DataLoader(
    test_reconstruction_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True, sampler=test_reconstruction_sampler
)


Loading dataset:   0%|                                                                                                                                                                         | 0/4 [00:00<?, ?it/s]

Loading dataset: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.18it/s]


Unhealthy (lesions test set)

In [8]:
large_group = ['sub-1010', 'sub-1013', 'sub-1015', 'sub-1039', 'sub-1041', 'sub-1071', 'sub-1073', 'sub-1086', 'sub-1102', 'sub-1115', 'sub-113', 'sub-1149', 'sub-1164', 'sub-1165', 'sub-116', 'sub-1204', 'sub-1209', 'sub-1213', 'sub-1215', 'sub-1227', 'sub-1246', 'sub-1258', 'sub-127', 'sub-1292', 'sub-1312', 'sub-1314', 'sub-1320', 'sub-1323', 'sub-1354', 'sub-1355', 'sub-1358', 'sub-1364', 'sub-1366', 'sub-1373', 'sub-1386', 'sub-1395', 'sub-1409', 'sub-1410', 'sub-1422', 'sub-1432', 'sub-1445', 'sub-1447', 'sub-1475', 'sub-1478', 'sub-1480', 'sub-1483', 'sub-1485', 'sub-1488', 'sub-1508', 'sub-1517', 'sub-1552', 'sub-1554', 'sub-1555', 'sub-1569', 'sub-1598', 'sub-1634', 'sub-1656', 'sub-1670', 'sub-1719', 'sub-1725', 'sub-1727', 'sub-1736', 'sub-174', 'sub-185', 'sub-190', 'sub-196', 'sub-198', 'sub-221', 'sub-235', 'sub-241', 'sub-247', 'sub-249', 'sub-260', 'sub-264', 'sub-278', 'sub-294', 'sub-303', 'sub-321', 'sub-326', 'sub-335', 'sub-339', 'sub-341', 'sub-343', 'sub-345', 'sub-366', 'sub-370', 'sub-374', 'sub-398', 'sub-3', 'sub-400', 'sub-422', 'sub-42', 'sub-432', 'sub-433', 'sub-443', 'sub-446', 'sub-457', 'sub-463', 'sub-466', 'sub-47', 'sub-494', 'sub-501', 'sub-505', 'sub-512', 'sub-517', 'sub-521', 'sub-525', 'sub-529', 'sub-530', 'sub-53', 'sub-543', 'sub-563', 'sub-56', 'sub-572', 'sub-613', 'sub-631', 'sub-634', 'sub-638', 'sub-651', 'sub-652', 'sub-661', 'sub-682', 'sub-692', 'sub-698', 'sub-699', 'sub-707', 'sub-724', 'sub-751', 'sub-760', 'sub-761', 'sub-768', 'sub-776', 'sub-791', 'sub-803', 'sub-823', 'sub-843', 'sub-844', 'sub-861', 'sub-865', 'sub-866', 'sub-877', 'sub-881', 'sub-8', 'sub-917', 'sub-937', 'sub-939', 'sub-942', 'sub-946', 'sub-952', 'sub-959', 'sub-95', 'sub-960', 'sub-968']

test_anomaly_images = sorted(glob.glob(ROOT_DIR+"datasets/final_soop_dataset_small/adc_registered/*.nii.gz"))

tests_anomaly_masks = glob.glob(ROOT_DIR+"datasets/final_soop_dataset_small/masks_combined_registered/*.nii.gz")

basic_affine = nib.load(test_anomaly_images[0]).affine

images_to_exclude = []
with open(ROOT_DIR+"AnoDiffExperiments/data_splits_lists/final_soop_dataset_small/exclude.csv", 'r') as f:
    for line in f:
        images_to_exclude.append(line.strip())

with open(ROOT_DIR+"AnoDiffExperiments/data_splits_lists/final_soop_dataset_small/exclude_non_axial_thick_slices.csv", 'r') as f:
    for line in f:
        images_to_exclude.append(line.strip())

test_anomaly_transforms = define_instance(args, "val_transforms")

test_masks_transforms = transforms.Compose(
    [
        transforms.LoadImage(),
        transforms.EnsureChannelFirst(),
        transforms.ResizeWithPadOrCrop(spatial_size=(args.image_size, args.image_size, args.image_size)),
        custom_transforms.SetBackgroundToZero()
    ]
)

test_anomaly_large_images = [path for path in test_anomaly_images if os.path.basename(path).split('.')[0] not in images_to_exclude and os.path.basename(path).split('.')[0] in large_group]        
large_group_masks = [path for path in tests_anomaly_masks if os.path.basename(path).split('.')[0] not in images_to_exclude and os.path.basename(path).split('.')[0] in large_group]

test_anomaly_large_images = sorted(test_anomaly_large_images, key=lambda x: os.path.basename(x).split('.')[0])
large_group_masks = sorted(large_group_masks, key=lambda x: os.path.basename(x).split('.')[0])

test_anomaly_large_ds = CacheDataset(data=test_anomaly_large_images, transform=test_anomaly_transforms)

test_anomaly_large_loader = DataLoader(       # The second 50% is used to compute the final IOU and DICE metrics with these best values.
    test_anomaly_large_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True
)

test_masks_large_ds = CacheDataset(data=large_group_masks, transform=test_masks_transforms)

test_masks_large_loader = DataLoader(
    test_masks_large_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True
)

Loading dataset: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 80/80 [00:16<00:00,  4.73it/s]


### Models

In [9]:

# ----------------- MODEL, OPTIMIZER, LOSS, LR SCHEDULER ----------------- #
# Define Autoencoder KL network and diffusion model
# Load Autoencoder KL network


autoencoder = define_instance(args, "autoencoder_def").to(device)

trained_g_path = os.path.join(MODELS_DIR, f"{SUB_EXPERIMENT_NAME}_autoencoder.pt")

map_location = {"cuda:%d" % 0: "cuda:%d" % rank}
autoencoder.load_state_dict(torch.load(trained_g_path, map_location=map_location, weights_only=True))
print(f"Rank {rank}: Load trained autoencoder from {trained_g_path}")

Rank 0: Load trained autoencoder from /bettik/PROJECTS/pr-gin5_aini/fehrdelt/AnoDiffExperiments/experiment_2/exp_2_7/models/exp_2_7_autoencoder.pt


In [10]:
if rank==0:
        os.makedirs(ROOT_DIR+f"AnoDiffExperiments/tensorboard/{SUB_EXPERIMENT_NAME}", exist_ok=True)
        writer = SummaryWriter(ROOT_DIR+f"AnoDiffExperiments/tensorboard/{SUB_EXPERIMENT_NAME}")


# Compute Scaling factor
# As mentioned in Rombach et al. [1] Section 4.3.2 and D.1, the signal-to-noise ratio (induced by the scale of the latent space) can affect the results obtained with the LDM,
# if the standard deviation of the latent space distribution drifts too much from that of a Gaussian.
# For this reason, it is best practice to use a scaling factor to adapt this standard deviation.
# _Note: In case where the latent space is close to a Gaussian distribution, the scaling factor will be close to one,
# and the results will not differ from those obtained when it is not used._

with torch.no_grad():
    with autocast("cuda", enabled=True):
        check_data = first(test_reconstruction_loader)
        z = autoencoder.encode_stage_2_inputs(check_data.to(device))
        if rank == 0:
            print(f"Latent feature shape {z.shape}")
            for axis in range(3):
                writer.add_image(
                    "train_img_" + str(axis),
                    visualize_one_slice_in_3d_image(check_data[0, 0, ...], axis).transpose([2, 1, 0]),
                    1,
                )
            print(f"Scaling factor set to {1/torch.std(z)}")

scale_factor = 1 / torch.std(z)
print(f"Rank {rank}: local scale_factor: {scale_factor}")
if ddp_bool:
    dist.barrier()
    dist.all_reduce(scale_factor, op=torch.distributed.ReduceOp.AVG)
print(f"Rank {rank}: final scale_factor -> {scale_factor}")

Latent feature shape torch.Size([4, 8, 32, 32, 32])
Scaling factor set to 1.0047698020935059
Rank 0: local scale_factor: 1.0047698020935059
Rank 0: final scale_factor -> 1.0047698020935059


In [11]:
# Define Diffusion Model
unet = define_instance(args, "diffusion_network_def").to(device)

trained_diffusion_path = os.path.join(MODELS_DIR, "diffusion_unet.pt")
trained_diffusion_path_last = os.path.join(MODELS_DIR, "diffusion_unet_last.pt")

unet.load_state_dict(torch.load(trained_diffusion_path, map_location="cuda:0"))
unet.eval()

scheduler = DDPMScheduler(
    num_train_timesteps=args.noise["num_timesteps_full_noise"],
    schedule="scaled_linear_beta",
    beta_start=args.noise["beta_start"],
    beta_end=args.noise["beta_end"],
)

if ddp_bool:
    autoencoder = DDP(autoencoder, device_ids=[device], output_device=rank, find_unused_parameters=True)
    unet = DDP(unet, device_ids=[device], output_device=rank, find_unused_parameters=True)

# We define the inferer using the scale factor:
inferer = LatentDiffusionInferer(scheduler, scale_factor=scale_factor)


In [12]:
optimizer_diff = torch.optim.Adam(params=unet.parameters(), lr=1e-5 * world_size)
lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer_diff, milestones=[100, 1000], gamma=0.1)


### Healthy reconstruction test

In [17]:
infer_timesteps = 30

for step, batch in enumerate(test_reconstruction_loader):

    if step>0:break  # only one batch for checking

    images = batch.to(device)
    optimizer_diff.zero_grad(set_to_none=True)

    with torch.no_grad():
        with autocast("cuda", enabled=True):

            latents = autoencoder.encode_stage_2_inputs(images)    

            # Add noise to latents
            noise = torch.randn_like(latents).to(device)
            timesteps = torch.randint(0, infer_timesteps, (latents.shape[0],), device=device).long()
            noisy_latents = scheduler.add_noise(latents, noise, timesteps)
            
            # Denoise completely using the UNet
            scheduler.set_timesteps(scheduler.num_train_timesteps)
            current_latents = noisy_latents * scale_factor
            
            for t in tqdm(range(infer_timesteps-1, -1, -1)):
                noise_pred = unet(current_latents, timesteps=torch.tensor([t], device=device).expand(latents.shape[0]))
                current_latents, _ = scheduler.step(noise_pred, t, current_latents)
            
            # Decode the denoised latents
            current_latents = current_latents / scale_factor
            reconstructed_images = autoencoder.decode(current_latents)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 53.73it/s]


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Get middle slices indices
mid_axial = images.shape[4] // 2
mid_coronal = images.shape[3] // 2
mid_sagittal = images.shape[2] // 2

# Original images (first sample in batch)
axes[0, 0].imshow(images[0, 0, :, :, mid_axial].cpu().numpy(), cmap='gray')
axes[0, 0].set_title('Original - Axial')
axes[0, 1].imshow(images[0, 0, :, mid_coronal, :].cpu().numpy(), cmap='gray')
axes[0, 1].set_title('Original - Coronal')
axes[0, 2].imshow(images[0, 0, mid_sagittal, :, :].cpu().numpy(), cmap='gray')
axes[0, 2].set_title('Original - Sagittal')

# Reconstructed images
axes[1, 0].imshow(reconstructed_images[0, 0, :, :, mid_axial].cpu().numpy(), cmap='gray')
axes[1, 0].set_title('Reconstructed - Axial')
axes[1, 1].imshow(reconstructed_images[0, 0, :, mid_coronal, :].cpu().numpy(), cmap='gray')
axes[1, 1].set_title('Reconstructed - Coronal')
axes[1, 2].imshow(reconstructed_images[0, 0, mid_sagittal, :, :].cpu().numpy(), cmap='gray')
axes[1, 2].set_title('Reconstructed - Sagittal')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle(f"{infer_timesteps} sampling steps (gaussian noise)", fontsize=16)

plt.tight_layout()
plt.show()

#### 150 sampling steps

In [ ]:
infer_timesteps = 150

for step, batch in enumerate(test_reconstruction_loader):

    if step>0:break  # only one batch for checking

    images = batch.to(device)
    optimizer_diff.zero_grad(set_to_none=True)

    with torch.no_grad():
        with autocast("cuda", enabled=True):

            latents = autoencoder.encode_stage_2_inputs(images)    

            # Add noise to latents
            noise = torch.randn_like(latents).to(device)
            timesteps = torch.randint(0, scheduler.num_train_timesteps, (latents.shape[0],), device=device).long()
            noisy_latents = scheduler.add_noise(latents, noise, timesteps) 
            
            # Denoise completely using the UNet
            scheduler.set_timesteps(scheduler.num_train_timesteps)
            current_latents = noisy_latents * scale_factor
            
            for t in tqdm(range(infer_timesteps-1, -1, -1)):
                noise_pred = unet(current_latents, timesteps=torch.tensor([t], device=device).expand(latents.shape[0]))
                current_latents, _ = scheduler.step(noise_pred, t, current_latents)
            
            # Decode the denoised latents
            current_latents = current_latents / scale_factor
            reconstructed_images = autoencoder.decode(current_latents)


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 150/150 [00:01<00:00, 81.83it/s]


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Get middle slices indices
mid_axial = images.shape[4] // 2
mid_coronal = images.shape[3] // 2
mid_sagittal = images.shape[2] // 2

# Original images (first sample in batch)
axes[0, 0].imshow(images[0, 0, :, :, mid_axial].cpu().numpy(), cmap='gray')
axes[0, 0].set_title('Original - Axial')
axes[0, 1].imshow(images[0, 0, :, mid_coronal, :].cpu().numpy(), cmap='gray')
axes[0, 1].set_title('Original - Coronal')
axes[0, 2].imshow(images[0, 0, mid_sagittal, :, :].cpu().numpy(), cmap='gray')
axes[0, 2].set_title('Original - Sagittal')

# Reconstructed images
axes[1, 0].imshow(reconstructed_images[0, 0, :, :, mid_axial].cpu().numpy(), cmap='gray')
axes[1, 0].set_title('Reconstructed - Axial')
axes[1, 1].imshow(reconstructed_images[0, 0, :, mid_coronal, :].cpu().numpy(), cmap='gray')
axes[1, 1].set_title('Reconstructed - Coronal')
axes[1, 2].imshow(reconstructed_images[0, 0, mid_sagittal, :, :].cpu().numpy(), cmap='gray')
axes[1, 2].set_title('Reconstructed - Sagittal')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle(f"{infer_timesteps} sampling steps (gaussian noise)", fontsize=16)

plt.tight_layout()
plt.show()

### Unhealthy anomaly detection

In [48]:
infer_timesteps = 30

for step, batch in enumerate(test_anomaly_large_loader):

    if step>0:break  # only one batch for checking

    images = batch.to(device)
    optimizer_diff.zero_grad(set_to_none=True)

    with torch.no_grad():
        with autocast("cuda", enabled=True):

            latents = autoencoder.encode_stage_2_inputs(images)    

            # Add noise to latents
            noise = torch.randn_like(latents).to(device)
            timesteps = torch.randint(0, scheduler.num_train_timesteps, (latents.shape[0],), device=device).long()
            noisy_latents = scheduler.add_noise(latents, noise, timesteps)
            
            # Denoise completely using the UNet
            scheduler.set_timesteps(scheduler.num_train_timesteps)
            current_latents = noisy_latents * scale_factor
            
            for t in tqdm(range(infer_timesteps-1, -1, -1)):
                noise_pred = unet(current_latents, timesteps=torch.tensor([t], device=device).expand(latents.shape[0]))
                current_latents, _ = scheduler.step(noise_pred, t, current_latents)
            
            # Decode the denoised latents
            current_latents = current_latents / scale_factor
            reconstructed_images = autoencoder.decode(current_latents)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 83.87it/s]


In [51]:
image_to_visualize = 0

normalized_reconstructed_images = scale_intensity_from_histogram_peak(reconstructed_images[image_to_visualize,0,...], target_value=2.0/7.0)



In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# Get middle slices indices
mid_axial = images.shape[4] // 2
mid_coronal = images.shape[3] // 2
mid_sagittal = images.shape[2] // 2

# Original images (first sample in batch)
axes[0, 0].imshow(images[image_to_visualize, 0, :, :, mid_axial].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Original - Axial')
axes[0, 1].imshow(images[image_to_visualize, 0, :, mid_coronal, :].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Original - Coronal')
axes[0, 2].imshow(images[image_to_visualize, 0, mid_sagittal, :, :].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 2].set_title('Original - Sagittal')



# Reconstructed images
axes[1, 0].imshow(normalized_reconstructed_images[:, :, mid_axial].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title('Reconstructed - Axial')
axes[1, 1].imshow(normalized_reconstructed_images[:, mid_coronal, :].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 1].set_title('Reconstructed - Coronal')
axes[1, 2].imshow(normalized_reconstructed_images[mid_sagittal, :, :].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 2].set_title('Reconstructed - Sagittal')

# Difference images
diff = torch.abs(images[image_to_visualize,0,...] - normalized_reconstructed_images)
axes[2, 0].imshow(diff[:, :, mid_axial].cpu().numpy(), cmap='hot')
axes[2, 0].set_title('Difference - Axial')
axes[2, 1].imshow(diff[:, mid_coronal, :].cpu().numpy(), cmap='hot')
axes[2, 1].set_title('Difference - Coronal')
axes[2, 2].imshow(diff[mid_sagittal, :, :].cpu().numpy(), cmap='hot')
axes[2, 2].set_title('Difference - Sagittal')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle(f"{infer_timesteps} sampling steps (gaussian noise)", fontsize=16)

plt.tight_layout()
plt.show()

modifs timesteps


In [ ]:
infer_timesteps = 30

for step, batch in enumerate(test_anomaly_large_loader):

    if step>0:break  # only one batch for checking

    images = batch.to(device)
    optimizer_diff.zero_grad(set_to_none=True)

    with torch.no_grad():
        with autocast("cuda", enabled=True):

            latents = autoencoder.encode_stage_2_inputs(images)    

            # Add noise to latents
            noise = torch.randn_like(latents).to(device)
            timesteps = torch.randint(0, scheduler.num_train_timesteps, (latents.shape[0],), device=device).long()
            
            noisy_latents = scheduler.add_noise(latents, noise, timesteps)
            
            # Denoise completely using the UNet
            scheduler.set_timesteps(scheduler.num_train_timesteps)
            
            current_latents = noisy_latents * scale_factor
            
            for t in tqdm(range(infer_timesteps-1, -1, -1)):
                noise_pred = unet(current_latents, timesteps=torch.tensor([t], device=device).expand(latents.shape[0]))
                current_latents, _ = scheduler.step(noise_pred, t, current_latents)
            
            # Decode the denoised latents
            current_latents = current_latents / scale_factor
            reconstructed_images = autoencoder.decode(current_latents)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 85.44it/s]


In [17]:
image_to_visualize = 0

normalized_reconstructed_images = scale_intensity_from_histogram_peak(reconstructed_images[image_to_visualize,0,...], target_value=2.0/7.0)



In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# Get middle slices indices
mid_axial = images.shape[4] // 2
mid_coronal = images.shape[3] // 2
mid_sagittal = images.shape[2] // 2

# Original images (first sample in batch)
axes[0, 0].imshow(images[image_to_visualize, 0, :, :, mid_axial].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Original - Axial')
axes[0, 1].imshow(images[image_to_visualize, 0, :, mid_coronal, :].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Original - Coronal')
axes[0, 2].imshow(images[image_to_visualize, 0, mid_sagittal, :, :].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 2].set_title('Original - Sagittal')



# Reconstructed images
axes[1, 0].imshow(normalized_reconstructed_images[:, :, mid_axial].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title('Reconstructed - Axial')
axes[1, 1].imshow(normalized_reconstructed_images[:, mid_coronal, :].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 1].set_title('Reconstructed - Coronal')
axes[1, 2].imshow(normalized_reconstructed_images[mid_sagittal, :, :].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 2].set_title('Reconstructed - Sagittal')

# Difference images
diff = torch.abs(images[image_to_visualize,0,...] - normalized_reconstructed_images)
axes[2, 0].imshow(diff[:, :, mid_axial].cpu().numpy(), cmap='hot')
axes[2, 0].set_title('Difference - Axial')
axes[2, 1].imshow(diff[:, mid_coronal, :].cpu().numpy(), cmap='hot')
axes[2, 1].set_title('Difference - Coronal')
axes[2, 2].imshow(diff[mid_sagittal, :, :].cpu().numpy(), cmap='hot')
axes[2, 2].set_title('Difference - Sagittal')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle(f"{infer_timesteps} sampling steps (gaussian noise)", fontsize=16)

plt.tight_layout()
plt.show()